In [2]:
import pandas as pd
import glob

In [3]:
# Creating dataframe with Listing Files
listing_files = glob.glob("Listing/CRMLSListing*.csv")
listed_df = pd.concat([pd.read_csv(f)for f in listing_files], ignore_index=True)

# Creating dataframe with Sold Files
sold_files = glob.glob("Sold/CRMLSSold*.csv")
sold_df = pd.concat([pd.read_csv(s)for s in sold_files], ignore_index=True)

/var/folders/l4/hlg0lrhx2_79czgk4q732ylw0000gq/T/ipykernel_50334/2781649819.py:7: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  sold_df = pd.concat([pd.read_csv(s)for s in sold_files], ignore_index=True)
/var/folders/l4/hlg0lrhx2_79czgk4q732ylw0000gq/T/ipykernel_50334/2781649819.py:7: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  sold_df = pd.concat([pd.read_csv(s)for s in sold_files], ignore_index=True)
/var/folders/l4/hlg0lrhx2_79czgk4q732ylw0000gq/T/ipykernel_50334/2781649819.py:7: DtypeWarning: Columns (2,36,39,56,74) have mixed types. Specify dtype option on import or set low_memory=False.
  sold_df = pd.concat([pd.read_csv(s)for s in sold_files], ignore_index=True)


In [4]:

listed_df = listed_df[listed_df["PropertyType"] == "Residential"]
sold_df = sold_df[sold_df["PropertyType"] == "Residential"]


null_listed_pct = (listed_df.isnull().sum() / len(listed_df)) * 100
null_sold_pct = (sold_df.isnull().sum() / len(sold_df)) * 100


high_missing_listed = null_listed_pct[null_listed_pct > 90]
high_missing_sold = null_sold_pct[null_sold_pct > 90]


listed_df = listed_df.drop(columns=high_missing_listed.index)
sold_df = sold_df.drop(columns=high_missing_sold.index)

listed_df.to_csv("listings_residential_cleaned.csv", index=False)
sold_df.to_csv("sold_residential_cleaned.csv", index=False)

In [5]:
#WEEK 3:

sold_df = pd.read_csv("sold_residential_cleaned.csv")
listings_df = pd.read_csv("listings_residential_cleaned.csv")


/var/folders/l4/hlg0lrhx2_79czgk4q732ylw0000gq/T/ipykernel_50334/3048225977.py:3: DtypeWarning: Columns (0,1,7,51,63,64,65) have mixed types. Specify dtype option on import or set low_memory=False.
  sold_df = pd.read_csv("sold_residential_cleaned.csv")
/var/folders/l4/hlg0lrhx2_79czgk4q732ylw0000gq/T/ipykernel_50334/3048225977.py:4: DtypeWarning: Columns (2,69) have mixed types. Specify dtype option on import or set low_memory=False.
  listings_df = pd.read_csv("listings_residential_cleaned.csv")


In [6]:
# Step 1 – Fetch the mortgage rate data from FRED
import pandas as pd
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

In [7]:
# Step 2 – Resample weekly rates to monthly averages
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (mortgage.groupby('year_month')['rate_30yr_fixed'].mean().reset_index())

In [8]:
# Step 3 – Create a matching year_month key on the MLS datasets
# Sold dataset — key off CloseDate
sold_df['year_month'] = pd.to_datetime(sold_df['CloseDate']).dt.to_period('M')
# Listings dataset — key off ListingContractDate
listings_df['year_month'] = pd.to_datetime(listings_df['ListingContractDate']).dt.to_period('M')

In [9]:
# Step 4 – Merge
sold_with_rates = sold_df.merge(mortgage_monthly, on='year_month', how='left')
listings_with_rates = listings_df.merge(mortgage_monthly, on='year_month', how='left')

In [10]:
# Step 5 – Validate the merge
# Check for any unmatched rows (rate should not be null)
sold_nulls = sold_with_rates['rate_30yr_fixed'].isnull().sum()
listings_nulls = listings_with_rates['rate_30yr_fixed'].isnull().sum()

print("Sold missing mortgage rates:", sold_nulls)
print("Listings missing mortgage rates:", listings_nulls)

Sold missing mortgage rates: 0
Listings missing mortgage rates: 0


In [11]:
# Preview
print(sold_with_rates[['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']].head())
print(listings_with_rates[['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']].head())

    CloseDate year_month  ClosePrice  rate_30yr_fixed
0  2024-09-12    2024-09   1090000.0             6.18
1  2024-09-30    2024-09   1995000.0             6.18
2  2024-09-30    2024-09   2340000.0             6.18
3  2024-09-30    2024-09    984000.0             6.18
4  2024-09-30    2024-09   1225000.0             6.18
  CloseDate year_month  ClosePrice  rate_30yr_fixed
0       NaN    2024-05         NaN             7.06
1       NaN    2024-05         NaN             7.06
2       NaN    2024-05         NaN             7.06
3       NaN    2024-05         NaN             7.06
4       NaN    2024-05         NaN             7.06


In [12]:
sold_with_rates.to_csv("soldRates.csv", index=False)
listings_with_rates.to_csv("listingsRates.csv", index=False)